In [0]:
# ===============================
# SILVER LAYER NOTEBOOK
# ===============================

import uuid
from pyspark.sql.functions import lit, current_timestamp

# -------------------------------
# 1️⃣ Pipeline Execution Metadata
# -------------------------------

run_id = str(uuid.uuid4())
job_id = "manual_run"
job_name = "silver_layer_job"
env = "dev"

# If running outside Git repo
git_commit = "not_in_repo"
git_branch = "not_in_repo"
git_repo = "not_in_repo"

print("Run ID:", run_id)

# -------------------------------
# 2️⃣ Insert Pipeline Start Record
# -------------------------------

spark.sql(f"""
INSERT INTO pipeline_runs (
    run_id,
    job_id,
    job_name,
    env,
    git_commit,
    git_branch,
    git_repo,
    start_time,
    end_time,
    status
)
VALUES (
    '{run_id}',
    '{job_id}',
    '{job_name}',
    '{env}',
    '{git_commit}',
    '{git_branch}',
    '{git_repo}',
    current_timestamp(),
    NULL,
    'RUNNING'
)
""")

# -------------------------------
# 3️⃣ Capture Bronze Input Version
# -------------------------------

history = spark.sql("DESCRIBE HISTORY bronze_orders")
input_version = history.select("version").first()[0]

# -------------------------------
# 4️⃣ Transform Bronze → Silver
# -------------------------------

df = spark.read.table("bronze_orders") \
    .filter("status = 'completed'")

df_out = df \
    .withColumn("run_id", lit(run_id)) \
    .withColumn("git_commit", lit(git_commit)) \
    .withColumn("processed_at", current_timestamp())

df_out.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("silver_orders")

# -------------------------------
# 5️⃣ Capture Silver Output Version
# -------------------------------

output_version = spark.sql("DESCRIBE HISTORY silver_orders") \
    .select("version").first()[0]

# -------------------------------
# 6️⃣ Record Dataset Versions
# -------------------------------

spark.sql(f"""
INSERT INTO dataset_versions VALUES
('bronze_orders', '{input_version}', '{run_id}', current_timestamp())
""")

spark.sql(f"""
INSERT INTO dataset_versions VALUES
('silver_orders', '{output_version}', '{run_id}', current_timestamp())
""")

# -------------------------------
# 7️⃣ Record Lineage
# -------------------------------

spark.sql(f"""
INSERT INTO lineage_edges VALUES
('silver_orders', 'bronze_orders', '{input_version}', '{run_id}')
""")

# -------------------------------
# 8️⃣ Mark Pipeline SUCCESS
# -------------------------------

spark.sql(f"""
UPDATE pipeline_runs
SET end_time = current_timestamp(),
    status = 'SUCCESS'
WHERE run_id = '{run_id}'
""")

print("Silver layer completed successfully.")